# Comprehensive Exploratory Data Analysis (EDA) - MIMIC-IV Hospital Datasets
This notebook performs an extensive, automated data profiling of all the hospital datasets in the MIMIC-IV database.

**Objectives:**
1. **Data Discovery:** Automatically map out the schema, size, and contents of all 21 datasets.
2. **Dataset Linkage:** Create a comprehensive visual mapping of how datasets relate to each other through primary and foreign keys.
3. **Data Quality Profiling:** Uncover systemic missing values, anomalies, and outliers using rich, premium visualizations.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import glob
import os
import warnings
warnings.filterwarnings('ignore')

# Set a very rich and premium visualization theme
sns.set_theme(style="darkgrid", palette="mako", rc={"axes.facecolor": "#f8f9fa"})
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['lines.linewidth'] = 2.5


## 1. Automated Data Discovery & Dictionary
We will dynamically load all datasets in the `hosp` directory to understand their shapes, column structures, and memory footprint.

In [ ]:
hosp_path = 'input/data/mimic-iv-clinical-database-demo-2.2/hosp/'
all_files = glob.glob(os.path.join(hosp_path, "*.csv"))

# Filter out corrupted file names
all_files = [f for f in all_files if "," not in f]

summary_data = []
all_columns_map = {}

print(f"Discovered {len(all_files)} datasets. Generating Data Dictionary...")

for file in all_files:
    file_name = os.path.basename(file)
    try:
        df = pd.read_csv(file, low_memory=False)
        all_columns_map[file_name] = list(df.columns)
        
        summary_data.append({
            'Dataset': file_name,
            'Rows': df.shape[0],
            'Columns': df.shape[1],
            'Missing Cells': df.isnull().sum().sum(),
            'Missing %': (df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100 if df.shape[0] > 0 else 0,
            'Memory (MB)': df.memory_usage(deep=True).sum() / (1024 * 1024)
        })
    except Exception as e:
        print(f"Error reading {file_name}: {e}")

summary_df = pd.DataFrame(summary_data).sort_values(by='Rows', ascending=False)

# Premium visualization of the data dictionary using pandas styling
summary_df.style.background_gradient(cmap='viridis', subset=['Rows', 'Columns', 'Memory (MB)'])\
                .background_gradient(cmap='OrRd', subset=['Missing %'])\
                .format({'Missing %': "{:.2f}%", 'Memory (MB)': "{:.2f}"})


## 2. Comprehensive Visual of Dataset Relationships
To understand how these datasets relate, we will construct a network graph. The nodes will be the **Datasets** (blue) and the shared **Keys** (orange) like `subject_id` or `hadm_id`. Edges signify that a dataset contains that key.

```mermaid
erDiagram
    diagnoses_icd {
        int subject_id
        int hadm_id
        int icd_code
    }
    drgcodes {
        int subject_id
        int hadm_id
    }
    d_hcpcs {
    }
    d_icd_diagnoses {
        int icd_code
    }
    d_icd_procedures {
        int icd_code
    }
    d_labitems {
        int itemid
        int fluid
    }
    emar {
        int subject_id
        int hadm_id
        int emar_id
        int poe_id
        int pharmacy_id
        int enter_provider_id
    }
    emar_detail {
        int subject_id
        int emar_id
        int pharmacy_id
        int side
    }
    hcpcsevents {
        int subject_id
        int hadm_id
    }
    labevents {
        int labevent_id
        int subject_id
        int hadm_id
        int specimen_id
        int itemid
        int order_provider_id
    }
    microbiologyevents {
        int microevent_id
        int subject_id
        int hadm_id
        int micro_specimen_id
        int order_provider_id
        int spec_itemid
        int test_itemid
        int org_itemid
        int ab_itemid
    }
    omr {
        int subject_id
    }
    patients {
        int subject_id
    }
    pharmacy {
        int subject_id
        int hadm_id
        int pharmacy_id
        int poe_id
        int sliding_scale
    }
    poe {
        int poe_id
        int subject_id
        int hadm_id
        int discontinue_of_poe_id
        int discontinued_by_poe_id
        int order_provider_id
    }
    poe_detail {
        int poe_id
        int subject_id
    }
    prescriptions {
        int subject_id
        int hadm_id
        int pharmacy_id
        int poe_id
        int order_provider_id
    }
    procedures_icd {
        int subject_id
        int hadm_id
        int icd_code
    }
    provider {
        int provider_id
    }
    services {
        int subject_id
        int hadm_id
    }
    transfers {
        int subject_id
        int hadm_id
        int transfer_id
    }
    patients ||--o{ diagnoses_icd : "subject_id"
    patients ||--o{ drgcodes : "subject_id"
    patients ||--o{ emar : "subject_id"
    patients ||--o{ emar_detail : "subject_id"
    patients ||--o{ hcpcsevents : "subject_id"
    patients ||--o{ labevents : "subject_id"
    d_labitems ||--o{ labevents : "itemid"
    patients ||--o{ microbiologyevents : "subject_id"
    patients ||--o{ omr : "subject_id"
    patients ||--o{ pharmacy : "subject_id"
    patients ||--o{ poe : "subject_id"
    patients ||--o{ poe_detail : "subject_id"
    patients ||--o{ prescriptions : "subject_id"
    patients ||--o{ procedures_icd : "subject_id"
    patients ||--o{ services : "subject_id"
    patients ||--o{ transfers : "subject_id"
```

## 3. Deep Dive: Missing Data Analytics
Let's aggregate missing data metrics across all tables and visualize the most sparse fields.

In [ ]:
missing_data = []

for file in all_files:
    df = pd.read_csv(file, low_memory=False)
    for col in df.columns:
        missing_count = df[col].isnull().sum()
        if missing_count > 0:
            missing_data.append({
                'Dataset': os.path.basename(file),
                'Column': col,
                'Missing Count': missing_count,
                'Missing %': (missing_count / len(df)) * 100
            })

missing_df = pd.DataFrame(missing_data).sort_values(by='Missing %', ascending=False).head(20)

plt.figure(figsize=(14, 8))
ax = sns.barplot(data=missing_df, x='Missing %', y='Column', hue='Dataset', dodge=False, palette='flare')
plt.title("Top 20 Columns with Highest Missing Data Rates", pad=20)
plt.xlabel("Percentage Missing (%)")
plt.ylabel("Column Name")
# Move legend out of the plot
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Source Dataset')
plt.xlim(0, 100)
for p in ax.patches:
    width = p.get_width()
    if pd.notnull(width) and width > 0:
        plt.text(width + 1, p.get_y() + p.get_height() / 2, f'{width:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.show()


## 4. Anomaly and Outlier Detection (Lab Events)
We will focus on `labevents` combined with `d_labitems` to detect numeric outliers in frequently administered tests. This reveals out-of-range anomalies.

In [ ]:
# Load Lab Events and Lab Items dictionary
labevents = pd.read_csv(os.path.join(hosp_path, 'labevents.csv'), low_memory=False)
d_labitems = pd.read_csv(os.path.join(hosp_path, 'd_labitems.csv'), low_memory=False)

# Merge to get human-readable lab test names
lab_merged = pd.merge(labevents, d_labitems, on='itemid', how='left')

# Convert 'valuenum' to numeric, dropping NaNs
lab_merged['valuenum'] = pd.to_numeric(lab_merged['valuenum'], errors='coerce')
numeric_labs = lab_merged.dropna(subset=['valuenum'])

# Find the top 5 most frequently measured lab tests
top_labs = numeric_labs['label'].value_counts().head(5).index

filtered_labs = numeric_labs[numeric_labs['label'].isin(top_labs)]

# Use a Boxenplot (letter-value plot) which is excellent for large datasets with outliers
plt.figure(figsize=(16, 9))
sns.boxenplot(data=filtered_labs, x='label', y='valuenum', palette='cubehelix', hue='label', legend=False)
plt.title("Distribution and Outliers of Top 5 Lab Tests", pad=20)
plt.xlabel("Lab Test Label")
plt.ylabel("Measured Value (Numeric)")
plt.yscale('log') # Log scale because ranges differ wildly
plt.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Detect and print statistical anomalies for Potassium
potassium = numeric_labs[numeric_labs['label'].str.contains('Potassium', na=False, case=False)]
if not potassium.empty:
    mean_val = potassium['valuenum'].mean()
    std_val = potassium['valuenum'].std()
    outliers = potassium[(potassium['valuenum'] < mean_val - 3*std_val) | (potassium['valuenum'] > mean_val + 3*std_val)]
    print(f"Potassium Records: {len(potassium)}")
    print(f"Detected {len(outliers)} severe outliers (> 3 Std Dev from Mean)")
